In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

import tensorflow as tf
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model

from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.svm import SVC
import xgboost as xgb
from sklearn.decomposition import PCA
from sklearn.ensemble import VotingClassifier
from sklearn.utils import resample

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None)

In [ ]:
df = pd.read_csv('./dataset/brca_metabric_clinical_data.tsv', sep='\t')
df.to_csv('./dataset/brca.csv')

In [ ]:
df.drop(columns=['Study ID', 'Patient ID', 'Sample ID'], inplace=True)

In [ ]:
df.iloc[:, -2].value_counts()

In [ ]:
df.iloc[:, -1].value_counts()

In [ ]:
df.info()

In [ ]:
n_nan_columns = df.isnull().sum()
print(n_nan_columns.head())

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.iloc[:, -2].value_counts()

In [ ]:
df = df[df.iloc[:, -2] != 4.0]
df = df[df.iloc[:, -2] != 0.0]
df.iloc[:, -2].value_counts()

In [ ]:
X = df.iloc[:, :-2]
y = df.iloc[:, -2]

X = pd.get_dummies(X)

scaler = StandardScaler()
X = scaler.fit_transform(X)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_model = RandomForestClassifier()
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

print(classification_report(y_test, y_pred))

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf_model = RandomForestClassifier()
grid_search = GridSearchCV(estimator=rf_model, param_grid=param_grid, cv=5)

grid_search.fit(X_train, y_train)
best_rf = grid_search.best_estimator_
best_params = grid_search.best_params_


y_pred = best_rf.predict(X_test)

print("Best Parameters:", best_params)
print(classification_report(y_test, y_pred))

In [ ]:
svm_model = SVC(kernel='rbf')
svm_model.fit(X_train, y_train)

y_pred_svm = svm_model.predict(X_test)
print(classification_report(y_test, y_pred_svm))

In [ ]:
param_grid = {
    'kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
    'C': [0.1, 1, 10],
    'gamma': [0.1, 1, 10]
}

svm_model = SVC()
grid_search = GridSearchCV(estimator=svm_model, param_grid=param_grid, cv=5)

grid_search.fit(X_train, y_train)
best_svm = grid_search.best_estimator_
best_params = grid_search.best_params_

y_pred = best_svm.predict(X_test)

print("Best Parameters:", best_params)
print(classification_report(y_test, y_pred))

In [ ]:
kernels = ['linear', 'poly', 'rbf', 'sigmoid']

for kernel in kernels:
    svm_model = SVC(kernel=kernel, C=best_params['C'], gamma=best_params['gamma'])
    svm_model.fit(X_train, y_train)

    y_pred = svm_model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"{kernel}:", accuracy)

In [ ]:
param_grid = {
    'kernel': ['linear'],
    'C': [0.1, 1, 10],
    'gamma': [0.1, 1, 10]
}

svm_model = SVC(probability=True)
grid_search = GridSearchCV(estimator=svm_model, param_grid=param_grid, cv=5)

grid_search.fit(X_train, y_train)
best_svm = grid_search.best_estimator_
best_params = grid_search.best_params_

y_pred = best_svm.predict(X_test)

print("Best Parameters:", best_params)
print(classification_report(y_test, y_pred))

In [ ]:
le = LabelEncoder()
y_train_transformed = le.fit_transform(y_train)
y_test_transformed = le.transform(y_test)

xgb_model = xgb.XGBClassifier()
xgb_model.fit(X_train, y_train_transformed)

y_pred_xgb = xgb_model.predict(X_test)
print(classification_report(y_test_transformed, y_pred_xgb))

In [ ]:
param_grid = {
    'learning_rate': [0.1, 0.01, 0.001],
    'max_depth': [None, 3, 5, 7],
    'n_estimators': [10, 100, 200, 300]
}

le = LabelEncoder()
y_train_transformed = le.fit_transform(y_train)
y_test_transformed = le.transform(y_test)

xgb_model = xgb.XGBClassifier()
grid_search = GridSearchCV(estimator=xgb_model, param_grid=param_grid, cv=5)


grid_search.fit(X_train, y_train_transformed)
best_xgb = grid_search.best_estimator_
best_params = grid_search.best_params_

y_pred = best_xgb.predict(X_test)

print("Best Parameters:", best_params)
print(classification_report(y_test_transformed, y_pred))

In [ ]:
adaboost_model = AdaBoostClassifier()
adaboost_model.fit(X_train, y_train_transformed)

y_pred_adaboost = adaboost_model.predict(X_test)
print(classification_report(y_test_transformed, y_pred_adaboost))

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.1, 0.5, 1.0]
}

adaboost_model = AdaBoostClassifier()
grid_search = GridSearchCV(estimator=adaboost_model, param_grid=param_grid, cv=5)

grid_search.fit(X_train, y_train_transformed)
best_ada = grid_search.best_estimator_
best_params = grid_search.best_params_


y_pred = best_ada.predict(X_test)

print("Best Parameters:", best_params)
print(classification_report(y_test_transformed, y_pred))

In [ ]:
ensemble_model = VotingClassifier(estimators=[('xgb', best_xgb),
                                              ('rf', best_rf),
                                              ('svm', best_svm),
                                              ('ada', best_ada)],
                                              voting='soft')
ensemble_model.fit(X_train, y_train_transformed)

y_pred_ensemble = ensemble_model.predict(X_test)
print(classification_report(y_test_transformed, y_pred_ensemble))

In [ ]:
ensemble_model = VotingClassifier(estimators=[('xgb', best_xgb),
                                              ('rf', best_rf),
                                              ('svm', best_svm),
                                              ('ada', best_ada)],
                                              voting='hard')
ensemble_model.fit(X_train, y_train_transformed)

y_pred_ensemble = ensemble_model.predict(X_test)
print(classification_report(y_test_transformed, y_pred_ensemble))

In [ ]:
ensemble_hard = VotingClassifier(estimators=[('xgb', best_xgb),
                                              ('rf', best_rf),
                                              ('svm', best_svm),
                                              ('ada', best_ada)],
                                              voting='hard')
ensemble_soft = VotingClassifier(estimators=[('xgb', best_xgb),
                                              ('rf', best_rf),
                                              ('svm', best_svm),
                                              ('ada', best_ada)],
                                              voting='soft')

cv_scores_hard = cross_val_score(ensemble_hard, X, y, cv=10)
cv_scores_soft = cross_val_score(ensemble_soft, X, y, cv=10)

print("Accuracy (Hard Voting) for each fold:", cv_scores_hard)
print("Accuracy (Soft Voting) for each fold:", cv_scores_soft)

print("Mean Accuracy (Hard Voting):", cv_scores_hard.mean())
print("Mean Accuracy (Soft Voting):", cv_scores_soft.mean())

print("Max Accuracy (Hard Voting):", cv_scores_hard.max())
print("Max Accuracy (Soft Voting):", cv_scores_soft.max())

In [ ]:
from sklearn.model_selection import cross_val_predict

y_pred_ensemble_hard = cross_val_predict(ensemble_hard, X, y, cv=10)
classification_report_ensemble_hard = classification_report(y, y_pred_ensemble_hard)
print(classification_report_ensemble_hard)

In [ ]:
y_pred_ensemble_hard = cross_val_predict(ensemble_soft, X, y, cv=10)
classification_report_ensemble_soft = classification_report(y, y_pred_ensemble_hard)
print(classification_report_ensemble_soft)

In [ ]:
df_upsampled = df[df.iloc[:, -2] == 3.0]
df_upsampled = resample(df_upsampled, replace=True, n_samples=df[df.iloc[:, -2] != 3.0].shape[0], random_state=42)


df_combined = pd.concat([df[df.iloc[:, -2] != 3.0], df_upsampled])
X_upsampled = df_combined.iloc[:, :-2]
y_upsampled = df_combined.iloc[:, -2]

X_upsampled = pd.get_dummies(X_upsampled)
X_upsampled = scaler.transform(X_upsampled)

X_train_upsampled, X_test_upsampled, y_train_upsampled, y_test_upsampled = train_test_split(X_upsampled, y_upsampled, test_size=0.2, random_state=42)

ensemble_hard_upsampled = VotingClassifier(estimators=[('xgb', best_xgb),
                                              ('rf', best_rf),
                                              ('svm', best_svm),
                                              ('ada', best_ada)],
                                              voting='hard')

ensemble_soft_upsampled = VotingClassifier(estimators=[('xgb', best_xgb),
                                              ('rf', best_rf),
                                              ('svm', best_svm),
                                              ('ada', best_ada)],
                                              voting='soft')

cv_scores_hard = cross_val_score(ensemble_hard_upsampled, X_upsampled, y_upsampled, cv=10)
cv_scores_soft = cross_val_score(ensemble_soft_upsampled, X_upsampled, y_upsampled, cv=10)

print("Accuracy (Hard Voting) for each fold:", cv_scores_hard)
print("Mean Accuracy (Hard Voting):", cv_scores_hard.mean())
print("Max Accuracy (Hard Voting):", cv_scores_hard.max())

print("Accuracy (Soft Voting) for each fold:", cv_scores_soft)
print("Mean Accuracy (Soft Voting):", cv_scores_soft.mean())
print("Max Accuracy (Soft Voting):", cv_scores_soft.max())

In [ ]:
y_pred_ensemble_hard = cross_val_predict(ensemble_hard_upsampled, X_upsampled, y_upsampled, cv=10)
classification_report_ensemble_hard = classification_report(y_upsampled, y_pred_ensemble_hard)
print(classification_report_ensemble_hard)

In [ ]:
y_pred_ensemble_soft = cross_val_predict(ensemble_soft_upsampled, X_upsampled, y_upsampled, cv=10)
classification_report_ensemble_soft = classification_report(y_upsampled, y_pred_ensemble_soft)
print(classification_report_ensemble_soft)

In [ ]:
X_train_upsampled, X_test_upsampled, y_train_upsampled, y_test_upsampled = train_test_split(X_upsampled, y_upsampled, test_size=0.2, random_state=42)

ensemble_two_hard = VotingClassifier(estimators=[('xgb', best_xgb),('rf', best_rf)],voting='hard')
ensemble_two_soft = VotingClassifier(estimators=[('xgb', best_xgb),('rf', best_rf)],voting='soft')

cv_score_two_hard = cross_val_score(ensemble_two_hard, X_upsampled, y_upsampled, cv=10)
cv_score_two_soft = cross_val_score(ensemble_two_soft, X_upsampled, y_upsampled, cv=10)

print("Accuracy (Hard Voting) for each fold:", cv_score_two_hard)
print("Mean Accuracy (Hard Voting):", cv_score_two_hard.mean())
print("Max Accuracy (Hard Voting):", cv_score_two_hard.max())

print("Accuracy (Soft Voting) for each fold:", cv_score_two_soft)
print("Mean Accuracy (Soft Voting):", cv_score_two_soft.mean())
print("Max Accuracy (Soft Voting):", cv_score_two_soft.max())

In [ ]:
y_pred_ensemble_two_hard = cross_val_predict(ensemble_two_hard, X_upsampled, y_upsampled, cv=10)
classification_report_ensemble_two_hard = classification_report(y_upsampled, y_pred_ensemble_two_hard)
print(classification_report_ensemble_two_hard)

In [ ]:
y_pred_ensemble_two_soft = cross_val_predict(ensemble_two_soft, X_upsampled, y_upsampled, cv=10)
classification_report_ensemble_two_soft = classification_report(y_upsampled, y_pred_ensemble_two_soft)
print(classification_report_ensemble_two_soft)